# 025 — Agentic RAG with Planning
**Advanced RAG Series** | LLM decides WHEN and HOW to retrieve, iteratively

Covers: Plan-then-retrieve · Iterative retrieval · Tool-calling RAG agent · Query refinement


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
import os
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(f"Missing: {path}\nRun python create_datasets.py from project root.")

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, rag_text])
print(f"✓ Loaded {len(all_text):,} chars across 4 files")


✓ Loaded 22,486 chars across 4 files


In [4]:
# ── 1. Limitation of standard RAG ───────────────────────────────────────
print("STANDARD RAG: retrieve ONCE → answer")
print()
print("Problems:")
print("  1. Query may need multiple retrieval steps")
print("     'Compare BERT and GPT-3 on few-shot learning'")
print("     → needs to retrieve BERT info AND GPT-3 info separately")
print()
print("  2. First retrieval may reveal the REAL question")
print("     User: 'How does attention work in the best NLP model?'")
print("     First retrieve: discovers BERT is the answer → needs to retrieve BERT details")
print()
print("AGENTIC RAG: LLM plans → retrieves → evaluates → refines → answers")


STANDARD RAG: retrieve ONCE → answer

Problems:
  1. Query may need multiple retrieval steps
     'Compare BERT and GPT-3 on few-shot learning'
     → needs to retrieve BERT info AND GPT-3 info separately

  2. First retrieval may reveal the REAL question
     User: 'How does attention work in the best NLP model?'
     First retrieve: discovers BERT is the answer → needs to retrieve BERT details

AGENTIC RAG: LLM plans → retrieves → evaluates → refines → answers


In [5]:
# ── 2. Setup: vector store + retrieval tool ─────────────────────────────
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
docs_lc = splitter.create_documents([ai_text, ml_text, nlp_text, rag_text])

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.from_documents(docs_lc, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"✓ Vector store: {vectorstore.index.ntotal} vectors")


✓ Vector store: 85 vectors


In [6]:
# ── 3. Define tools for the agent ───────────────────────────────────────
from langchain_core.tools import tool

@tool
def search_knowledge_base(query: str) -> str:
    '''Search the AI/ML knowledge base. Use specific queries for best results.'''
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found."
    results = []
    for i, doc in enumerate(docs, 1):
        results.append(f"[Result {i}]\n{doc.page_content}")
    return "\n\n".join(results)

@tool
def evaluate_sufficiency(gathered_info: str, question: str) -> str:
    '''Evaluate if gathered information is sufficient to answer the question.'''
    prompt = (
        f"Question: {question}\n\n"
        f"Gathered information:\n{gathered_info}\n\n"
        "Is this information sufficient to answer the question completely? "
        "Reply with either:\n"
        "SUFFICIENT: [brief reason]\n"
        "INSUFFICIENT: [what is still missing]"
    )
    return llm.invoke(prompt).content

tools = [search_knowledge_base, evaluate_sufficiency]
tool_map = {t.name: t for t in tools}

print("Tools registered:")
for t in tools:
    print(f"  {t.name}: {t.description[:60]}...")


Tools registered:
  search_knowledge_base: Search the AI/ML knowledge base. Use specific queries for be...
  evaluate_sufficiency: Evaluate if gathered information is sufficient to answer the...


In [7]:
# ── 4. Agentic RAG loop (manual tool-calling) ────────────────────────────
import json
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def agentic_rag(question: str, max_steps: int = 6) -> str:
    messages = [
        HumanMessage(content=(
            f"Answer this question by searching the knowledge base as many times as needed.\n"
            f"Search for specific sub-topics separately to build a complete answer.\n"
            f"Question: {question}"
        ))
    ]

    llm_with_tools = llm_smart.bind_tools(tools)
    gathered_context = []

    for step in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            # No more tool calls — final answer
            return response.content

        # Execute each tool call
        for tc in response.tool_calls:
            tool_fn = tool_map[tc["name"]]
            result = tool_fn.invoke(tc["args"])
            gathered_context.append(result)
            messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))
            print(f"  Step {step+1}: called {tc['name']!r} → {len(result)} chars")

    return "Max steps reached."

print("Running agentic RAG...")
answer = agentic_rag("Compare BM25 retrieval and dense embedding retrieval. When should you use each?")
print("\n" + "=" * 50)
print("FINAL ANSWER:")
print("=" * 50)
print(answer)


Running agentic RAG...


  Step 1: called 'search_knowledge_base' → 905 chars
  Step 1: called 'search_knowledge_base' → 905 chars
  Step 1: called 'search_knowledge_base' → 1193 chars
  Step 1: called 'search_knowledge_base' → 1150 chars
  Step 1: called 'search_knowledge_base' → 1250 chars



FINAL ANSWER:
BM25 retrieval and dense embedding retrieval are two different approaches to information retrieval. BM25 retrieval uses term-frequency methods, which score documents based on term frequency and inverse document frequency, exceling at exact keyword matching and rare term retrieval. On the other hand, dense embedding retrieval uses embedding similarity to find semantically relevant documents, with bi-encoder models encoding queries and documents separately.

BM25 retrieval is suitable for use cases where exact keyword matching is important, such as in search engines where users are looking for specific information. It is also useful when dealing with rare terms, as it can effectively retrieve documents containing those terms.

Dense embedding retrieval, on the other hand, is suitable for use cases where semantic search is important, such as in question-answering systems or text generation tasks. It can effectively retrieve documents that are semantically relevant to the qu

In [8]:
# ── 5. LangGraph-based agentic RAG ──────────────────────────────────────
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm_smart, tools)

result = agent.invoke({
    "messages": [HumanMessage(content=(
        "What are the main differences between RAPTOR and standard hierarchical chunking? "
        "Search for both topics and compare them."
    ))]
})

final_message = result["messages"][-1]
print("LangGraph ReAct agent answer:")
print(final_message.content)
print(f"\nTotal messages in conversation: {len(result['messages'])}")


/home/saurabh/miniconda3/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


LangGraph ReAct agent answer:
The main differences between RAPTOR and standard hierarchical chunking are:

1. RAPTOR builds a tree of summaries at multiple levels of abstraction, enabling retrieval at different granularities, whereas standard hierarchical chunking splits documents into chunks based on fixed-size or semantic boundaries.
2. RAPTOR uses a recursive abstractive processing approach, whereas standard hierarchical chunking uses a hierarchical splitting approach.
3. RAPTOR preserves global context in chunk embeddings by embedding full documents and then splitting them, whereas standard hierarchical chunking may ignore semantics when splitting documents into chunks.

Overall, RAPTOR is a more advanced approach to document chunking that enables more fine-grained matching and retrieval, whereas standard hierarchical chunking is a simpler approach that may not capture the nuances of document structure and semantics.

Total messages in conversation: 6


In [9]:
# ── 6. Query planning before retrieval ──────────────────────────────────
PLAN_PROMPT = (
    "Break this question into 2-4 specific sub-queries to search a knowledge base.\n"
    "Each sub-query should target a specific concept.\n"
    "Output ONLY the sub-queries, one per line, no numbering.\n\n"
    "Question: {question}"
)

def plan_and_retrieve(question: str, top_k: int = 3) -> str:
    # Step 1: Plan
    plan = llm.invoke(PLAN_PROMPT.format(question=question)).content
    sub_queries = [q.strip() for q in plan.strip().split("\n") if q.strip()]
    print(f"Sub-queries planned:")
    for q in sub_queries:
        print(f"  - {q}")

    # Step 2: Retrieve for each sub-query
    all_chunks = {}
    for q in sub_queries:
        docs = retriever.invoke(q)
        for doc in docs:
            key = doc.page_content[:80]
            if key not in all_chunks:
                all_chunks[key] = doc.page_content

    # Step 3: Generate answer
    context = "\n\n---\n\n".join(list(all_chunks.values())[:top_k * len(sub_queries)])
    from langchain_core.prompts import ChatPromptTemplate
    prompt = ChatPromptTemplate.from_template(
        "Answer the question using the context below.\n"
        "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )
    return (prompt | llm).invoke({"context": context, "question": question}).content

answer = plan_and_retrieve("How does self-RAG differ from corrective RAG, and when should each be used?")
print("\nAnswer:")
print(answer)


Sub-queries planned:
  - What is the definition of self-RAG in the context of project management or risk assessment?
  - What is the definition of corrective RAG in the context of project management or risk assessment?
  - How does self-RAG differ from corrective RAG in terms of their application or usage?
  - When should self-RAG be used in a project or risk assessment scenario?
  - When should corrective RAG be used in a project or risk assessment scenario?



Answer:
Self-RAG and CRAG (Corrective RAG) differ in their evaluation and retrieval strategies. 

Self-RAG evaluates the model's own generated answer by checking if it is supported by the retrieved context. It decides when to retrieve, grades the retrieved documents for relevance, and checks the model's generated answer against the retrieved context.

CRAG, on the other hand, evaluates the retrieved documents and falls back to web search if the local knowledge base does not contain relevant information. This means CRAG is more focused on retrieving relevant information from external sources, whereas Self-RAG is more focused on evaluating the model's own performance.

Self-RAG should be used when the model needs to evaluate its own performance and ensure that its generated answer is supported by the retrieved context. This is particularly useful when the model is generating answers that require verifiable sources or up-to-date information.

CRAG should be used when the local knowledge 

In [10]:
# ── 7. Agentic RAG patterns summary ─────────────────────────────────────
print("AGENTIC RAG PATTERNS (2024-2025)")
print("=" * 45)
print()
print("1. PLAN-THEN-RETRIEVE")
print("   LLM decomposes question → sub-queries → retrieve each → merge → answer")
print("   Best for: multi-part questions")
print()
print("2. ITERATIVE RETRIEVAL (this notebook)")
print("   LLM retrieves → evaluates sufficiency → refines query → retrieves again")
print("   Best for: complex questions needing multiple hops")
print()
print("3. REACT AGENT (LangGraph)")
print("   LLM uses tools in a loop: Thought → Action → Observation → repeat")
print("   Best for: open-ended research tasks")
print()
print("4. SELF-RAG (notebook 010)")
print("   LLM decides IF retrieval is needed, grades retrieved docs, checks answer")
print("   Best for: mixed queries (some need retrieval, some don't)")
print()
print("Key libraries: langgraph.prebuilt.create_react_agent, langchain tools")


AGENTIC RAG PATTERNS (2024-2025)

1. PLAN-THEN-RETRIEVE
   LLM decomposes question → sub-queries → retrieve each → merge → answer
   Best for: multi-part questions

2. ITERATIVE RETRIEVAL (this notebook)
   LLM retrieves → evaluates sufficiency → refines query → retrieves again
   Best for: complex questions needing multiple hops

3. REACT AGENT (LangGraph)
   LLM uses tools in a loop: Thought → Action → Observation → repeat
   Best for: open-ended research tasks

4. SELF-RAG (notebook 010)
   LLM decides IF retrieval is needed, grades retrieved docs, checks answer
   Best for: mixed queries (some need retrieval, some don't)

Key libraries: langgraph.prebuilt.create_react_agent, langchain tools
